# Aula — Caderno 1, Capítulo 2: De Bachelier a Mandelbrot

**Livro-curso:** Econofísica Aplicada ao Mercado Financeiro\
**Autor:** Cristiano Alves\
**Duração:** 1h

### Objetivos da aula

1. Conhecer o passeio aleatório de Bachelier e a lei da raiz do tempo.
2. Entender por que Osborne passou a modelar o logaritmo do preço.
3. Reconhecer a crítica de Mandelbrot ao modelo gaussiano.
4. Medir, com dados, quão improvável a queda de 1987 seria sob a normal.

### Como usar este notebook

- Use o ambiente do Caderno 1 e faça antes a aula do Capítulo 1, que baixa os dados do
  S&P 500 e do Ibovespa (Listagem 1.1).
- Deixe o notebook na pasta `caderno-1/codigo/` ou na pasta `aulas/caderno-1/` do curso. A
  primeira célula encontra a pasta de código do caderno e passa a trabalhar nela.
- As Listagens 2.2 e 2.4 gravam as figuras do livro na pasta `figuras/` do caderno, e o
  notebook também as mostra.
- Execute as células **na ordem**, com `Shift+Enter`. Ao terminar, use *Kernel → Restart
  Kernel and Run All Cells* e confira se tudo roda do início ao fim.

### A história em uma tabela

A ideia de que os preços se movem ao acaso tem mais de um século. A aula segue quatro
momentos dessa história e, em cada um, reproduz no computador o argumento central.

| Ano | Quem | O quê |
|---|---|---|
| 1900 | Louis Bachelier | Tese *Théorie de la spéculation*: o preço como passeio aleatório |
| 1905 | Albert Einstein | O movimento browniano de partículas num fluido |
| 1959 | M. F. M. Osborne | O passeio aleatório vale para o logaritmo do preço |
| 1963 | Benoit Mandelbrot | Preços do algodão: caudas pesadas e distribuições estáveis de Lévy |
| 1965 | Eugene Fama | O mesmo excesso de valores extremos nas ações do Dow Jones |
| 1973 | Black e Scholes | Apreçamento de opções com o movimento browniano geométrico |
| 1987 | — | Segunda-Feira Negra: o S&P 500 cai 20,5% num dia |
| 1995 | Mantegna e Stanley | Leis de escala nas variações do S&P 500, na *Nature* |

### Preparação: a pasta de trabalho e os dados

In [ ]:
import os
import sys
from pathlib import Path


# Se a célula não encontrar a pasta do caderno, escreva aqui o caminho
# dela, por exemplo: PASTA_CODIGO = "/home/ana/curso/caderno-1/codigo"
PASTA_CODIGO = None


def localizar_codigo():
    """Procura a pasta de código do Caderno 1 (a que tem dados_caderno1.py):
    na pasta do notebook, na pasta de trabalho e nas pastas acima delas."""
    if PASTA_CODIGO:
        candidatos = [Path(PASTA_CODIGO)]
    else:
        bases = [Path.cwd()]
        if "__vsc_ipynb_file__" in globals():      # VS Code: pasta do notebook
            bases.insert(0, Path(__vsc_ipynb_file__).parent)
        candidatos = []
        for base in bases:
            for pasta in [base, *base.parents]:
                candidatos += [pasta, pasta / "codigo",
                               pasta / "caderno-1" / "codigo"]
    for candidato in candidatos:
        if (candidato / "dados_caderno1.py").is_file():
            return candidato.resolve()
    raise FileNotFoundError(
        f"dados_caderno1.py não encontrado a partir de {Path.cwd()}. Abra o "
        "notebook na pasta aulas/caderno-1 ou caderno-1/codigo do curso, ou "
        "escreva o caminho da pasta caderno-1/codigo em PASTA_CODIGO.")


pasta_codigo = localizar_codigo()
os.chdir(pasta_codigo)                      # os caminhos do caderno partem daqui
if str(pasta_codigo) not in sys.path:
    sys.path.insert(0, str(pasta_codigo))   # para importar dados_caderno1
Path("figuras").mkdir(exist_ok=True)        # as listagens gravam figuras aqui
print("Pasta de trabalho:", Path.cwd().name)

faltando = [nome for nome in ["sp500", "ibovespa"]
            if not any(Path("dados").glob(f"yahoo_{nome}_*.csv"))]
if faltando:
    print("Faltam os dados:", ", ".join(faltando),
          "- execute antes a Listagem 1.1 (aula do Capítulo 1).")
else:
    print("Dados do Caderno 1 encontrados.")

**Saída esperada:**

```text
Pasta de trabalho: codigo
Dados do Caderno 1 encontrados.
```

## 1. Bachelier e o passeio aleatório (1900)

Em março de 1900, Louis Bachelier defendeu em Paris a tese *Théorie de la spéculation*,
orientada por Henri Poincaré. Bachelier estudava os preços dos títulos da dívida pública
francesa e das opções sobre eles. A sua hipótese central: as variações de preço são a soma
de muitos pequenos choques independentes, de média zero. O preço faz um **passeio
aleatório**.

Para descrever esse movimento, Bachelier desenvolveu a matemática do que hoje se chama
**movimento browniano**, cinco anos antes do famoso artigo de Einstein sobre o movimento de
partículas num fluido. A tese, porém, permaneceu pouco conhecida entre os economistas até
ser redescoberta nos anos 1950.

A principal consequência do modelo é a **lei da raiz do tempo**: se cada dia soma um choque
independente de desvio-padrão $\sigma$, a dispersão do preço depois de $t$ dias é

$$\sigma\sqrt{t}, \quad \text{e não} \quad \sigma t.$$

Para quadruplicar a incerteza, é preciso esperar dezesseis vezes mais. A listagem confere a
lei com 10 000 trajetórias simuladas.

**Listagem 2.1.** O passeio aleatório de Bachelier e a lei da raiz do tempo.

In [ ]:
import numpy as np

# Passeio aleatório aritmético de Bachelier: o preço soma choques normais
rng = np.random.default_rng(seed=42)
dias, trajetorias, sigma = 400, 10_000, 1.0     # sigma em R$ por dia
choques = rng.normal(0, sigma, size=(dias, trajetorias))
precos = 100 + np.cumsum(choques, axis=0)

for t in [25, 100, 400]:
    observado = precos[t - 1].std()
    print(f"t = {t:3d} dias: desvio {observado:5.2f}   sigma*raiz(t) = "
          f"{sigma * np.sqrt(t):5.2f}")

**Saída esperada:**

```text
t =  25 dias: desvio  4.95   sigma*raiz(t) =  5.00
t = 100 dias: desvio 10.02   sigma*raiz(t) = 10.00
t = 400 dias: desvio 20.12   sigma*raiz(t) = 20.00
```

Com 25, 100 e 400 dias, o desvio observado é 5, 10 e 20: cada vez que o horizonte
quadruplica, a dispersão dobra. Você já usou essa lei no Caderno 0, ao anualizar a
volatilidade com $\sqrt{252}$.

### 1.1 A faixa da raiz do tempo

A figura mostra 30 das 10 000 trajetórias e a faixa $100 \pm 2\sigma\sqrt{t}$. A faixa
se abre cada vez mais devagar: é a curva da raiz quadrada. Como cada preço é a soma de
choques normais, ele também é normal, e cerca de 95% das trajetórias ficam dentro da faixa
em qualquer dia.

In [ ]:
import matplotlib.pyplot as plt

t = np.arange(1, dias + 1)
fig, ax = plt.subplots(figsize=(6.4, 3.0))
ax.plot(t, precos[:, :30], linewidth=0.5, alpha=0.7)
for sinal in (+2, -2):
    ax.plot(t, 100 + sinal * sigma * np.sqrt(t), color="black", linewidth=1.2)
ax.set_title("Trajetórias de Bachelier e a faixa 100 ± 2σ√t")
ax.set_xlabel("dias")
plt.show()

dentro = np.abs(precos[-1] - 100) < 2 * sigma * np.sqrt(dias)
print(f"No dia {dias}, {dentro.mean():.1%} das trajetórias estão dentro da faixa")

**Saída esperada:**

```text
No dia 400, 95.3% das trajetórias estão dentro da faixa
```

A figura aparece acima do texto: as trajetórias se espalham, e a faixa preta se abre como
uma raiz quadrada.

## 2. Osborne e o movimento browniano geométrico (1959)

O modelo de Bachelier tem um defeito: como soma choques em reais, e não em porcentagem, o
preço pode ficar negativo. Em 1959, o físico M. F. M. Osborne, analisando ações
norte-americanas, propôs que o passeio aleatório se aplica ao **logaritmo** do preço. As
variações passam a ser proporcionais ao preço, e o preço nunca cruza o zero. É o
**movimento browniano geométrico**, o mesmo modelo que você simulou no Capítulo 3 do
Caderno 0.

| | Aritmético (Bachelier) | Geométrico (Osborne) |
|---|---|---|
| O que faz o passeio | o preço | o logaritmo do preço |
| Choque de cada dia | em reais | em porcentagem |
| Pode ficar negativo? | sim | não |

**Listagem 2.2.** Passeio aritmético contra passeio geométrico em 25 anos.

In [ ]:
import matplotlib.pyplot as plt

# Mesma volatilidade inicial, 25 anos: aritmético (Bachelier) e
# geométrico (Osborne)
rng = np.random.default_rng(seed=7)
dias, n = 25 * 252, 20
choques = rng.normal(0, 0.02, size=(dias, n))       # 2% ao dia
aritmetico = 100 + np.cumsum(100 * choques, axis=0)
geometrico = 100 * np.exp(np.cumsum(choques, axis=0))

print("Trajetórias que ficaram negativas, de", n)
print("  aritmético:", (aritmetico.min(axis=0) < 0).sum())
print("  geométrico:", (geometrico.min(axis=0) < 0).sum())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6.4, 2.9), sharey=True)
anos = np.arange(dias) / 252
ax1.plot(anos, aritmetico, linewidth=0.6)
ax1.axhline(0, color="black", linewidth=0.8)
ax1.set_title("Aritmético (Bachelier)")
ax2.plot(anos, geometrico, linewidth=0.6)
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_title("Geométrico (Osborne)")
for ax in (ax1, ax2):
    ax.set_xlabel("anos")
ax1.set_ylabel("preço")
fig.savefig("figuras/fig_02_01_bachelier_osborne.pdf")

**Saída esperada:**

```text
Trajetórias que ficaram negativas, de 20
  aritmético: 10
  geométrico: 0
```

Vinte trajetórias de cada modelo ao longo de 25 anos, com choques de 2% ao dia. No passeio
aritmético, metade das trajetórias passou por preços negativos; no geométrico, nenhuma.

> **Nota.** O movimento browniano geométrico é a hipótese sobre os preços do modelo de
> Black e Scholes (1973) para o preço de opções, um dos resultados mais influentes das
> finanças. É por isso que o modelo gaussiano de retornos logarítmicos virou a referência
> de toda a área.

## 3. Mandelbrot e as caudas pesadas (1963)

Em 1963, Benoit Mandelbrot publicou um estudo das variações do preço do algodão nos Estados
Unidos. A conclusão contrariava o modelo gaussiano: variações extremas eram muito mais
frequentes do que a normal permite, e a distribuição dos preços tinha **caudas pesadas**.
Mandelbrot propôs substituí-la pelas distribuições **estáveis de Lévy**, que têm caudas em
lei de potência e, em certos casos, variância infinita. Pouco depois, a tese de Eugene Fama
encontrou o mesmo excesso de valores extremos nas ações do índice Dow Jones (1965).

A listagem mostra a força do argumento de Mandelbrot com o evento mais extremo da série do
S&P 500: a queda de 19 de outubro de 1987, a **Segunda-Feira Negra**. A pergunta é: com
tudo o que se sabia até a véspera, quão improvável essa queda parecia para o modelo
gaussiano?

O número de desvios-padrão é o **escore z**:

$$z = \frac{r - \bar r}{s},$$

em que $\bar r$ e $s$ são a média e o desvio-padrão do histórico até a véspera.

**Listagem 2.3.** A queda de 1987 medida em desvios-padrão.

In [ ]:
import pandas as pd
from scipy import stats

from dados_caderno1 import carregar_retornos

r_sp, _ = carregar_retornos()
historico = r_sp.loc[:"1987-10-16"]     # o que se sabia na véspera
queda = r_sp.loc["1987-10-19"]

z = (queda - historico.mean()) / historico.std()
print(f"Log-retorno em 19/10/1987: {queda:.1%}")
print(f"Desvio-padrão diário de 1950 até a véspera: {historico.std():.2%}")
print(f"A queda equivale a {z:.1f} desvios-padrão")
print(f"Probabilidade pela normal: {stats.norm.cdf(z):.1e}")

**Saída esperada:**

```text
Log-retorno em 19/10/1987: -22.9%
Desvio-padrão diário de 1950 até a véspera: 0.79%
A queda equivale a -29.1 desvios-padrão
Probabilidade pela normal: 7.3e-187
```

O log-retorno foi de −22,9%, ou 29 desvios-padrão abaixo da média histórica. Pela normal,
a probabilidade de uma queda assim num dia qualquer é da ordem de $10^{-187}$.

### 3.1 Quanto tempo esperar por uma queda assim?

Se um evento tem probabilidade $p$ por pregão, a espera média por ele é de $1/p$ pregões.
Compare com a idade do universo, cerca de 13,8 bilhões de anos:

In [ ]:
p_normal = stats.norm.cdf(z)
espera = 1 / p_normal                     # pregões, em média
idade_universo = 13.8e9 * 365.25          # dias
print(f"Espera média pela normal: {espera:.1e} pregões")
print(f"Idade do universo:        {idade_universo:.1e} dias")
print(f"A espera é {espera / idade_universo:.0e} vezes a idade do universo")

**Saída esperada:**

```text
Espera média pela normal: 1.4e+186 pregões
Idade do universo:        5.0e+12 dias
A espera é 3e+173 vezes a idade do universo
```

Sob a normal, a Segunda-Feira Negra simplesmente não poderia ter acontecido. Ela
aconteceu, e o modelo é que precisa ser revisto.

> **Armadilha comum.** Chamar eventos assim de "inesperados" ou de "eventos de 29 sigmas"
> como se fossem azar. Um evento de 29 desvios-padrão não diz que o mercado teve um dia de
> muito azar: diz que a distribuição normal é o modelo errado para as caudas. O Módulo 2
> mostra como medir as caudas de verdade.

## 4. A física estatística chega aos mercados (anos 1990)

As ideias de Mandelbrot não foram adotadas pela corrente principal das finanças, em parte
porque as distribuições estáveis tornam muitos cálculos difíceis. Nas décadas seguintes, a
teoria financeira se construiu sobre o modelo gaussiano. A figura mostra o que os dados
continuavam registrando enquanto isso.

**Listagem 2.4.** Os log-retornos diários do S&P 500 desde 1950.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.0))
ax.plot(r_sp.index, r_sp, linewidth=0.4, color="C0")
for data, rotulo in [("1987-10-19", "1987"), ("2008-10-15", "2008"),
                     ("2020-03-16", "2020")]:
    ax.annotate(rotulo, (pd.Timestamp(data), r_sp.loc[data]),
                xytext=(6, -2), textcoords="offset points", fontsize=8,
                color="C3")
ax.set_title("S&P 500: log-retornos diários, 1950 a 2025")
fig.savefig("figuras/fig_02_02_sp500_historia.pdf")
print(r_sp.nsmallest(3).round(3))

**Saída esperada:**

```text
Date
1987-10-19   -0.229
2020-03-16   -0.128
2020-03-12   -0.100
Name: ^GSPC, dtype: float64
```

Os três piores dias da série são o de 1987 e dois de março de 2020.

Nos anos 1990, com séries de alta frequência disponíveis e computadores capazes de
processá-las, físicos começaram a estudar esses dados com as ferramentas da física
estatística. Um marco é o artigo de Mantegna e Stanley de 1995 na revista *Nature*, que
analisou as variações do S&P 500 em diferentes escalas de tempo e encontrou leis de escala
na sua distribuição. A Econofísica nasceu desse encontro entre a crítica de Mandelbrot,
dados abundantes e os métodos da física dos sistemas complexos.

> **Contexto de mercado.** Repare na figura: os retornos grandes não estão espalhados ao
> acaso de 1950 a 2025. Eles se agrupam em alguns períodos, como 1987, 2008 e 2020. Esse
> agrupamento de volatilidade é o tema do Capítulo 4 e, com ferramentas mais finas, do
> Módulo 5.

## 5. Prática guiada: do argumento à ferramenta

A Listagem 2.3 responde a uma pergunta sobre um dia. Para fazer a mesma pergunta sobre
outros dias e outros índices, como pedem os exercícios, transforme-a numa função:

1. A função recebe a série de retornos e a data.
2. O histórico é tudo o que veio **antes** da data: `r.loc[:data - 1 dia]`.
3. Ela devolve o escore z e a probabilidade de um retorno tão baixo pela normal.
4. Confira que, para 19/10/1987, a função reproduz a Listagem 2.3.

In [ ]:
def desvios_na_vespera(r, data):
    """Escore z do retorno de um dia em relação ao histórico até a véspera,
    e a probabilidade de um retorno tão baixo pela normal."""
    data = pd.Timestamp(data)
    historico = r.loc[:data - pd.Timedelta(days=1)]
    z = (r.loc[data] - historico.mean()) / historico.std()
    return z, stats.norm.cdf(z)


z_1987, p_1987 = desvios_na_vespera(r_sp, "1987-10-19")
assert np.isclose(z_1987, z), "a função não reproduz a Listagem 2.3"
print(f"19/10/1987: {z_1987:.1f} desvios-padrão, probabilidade {p_1987:.1e}")

**Saída esperada:**

```text
19/10/1987: -29.1 desvios-padrão, probabilidade 7.3e-187
```

O `assert` garante que a função faz o mesmo cálculo da listagem. Se você alterar a função
e ela deixar de concordar, a célula para com um erro.

## 6. Exercícios

Os exercícios do Capítulo 2 do caderno. Cada um tem uma célula para a sua resposta e, logo
abaixo, uma **célula de conferência**, que responde *ok*, *confira* (com uma pista) ou
*ainda não resolvido*, sem interromper o notebook. As perguntas de interpretação têm uma
célula de texto.

Execute primeiro a célula a seguir, que define a conferência.

In [ ]:
def conferir(nome, teste):
    """Roda o teste de um exercício e diz se passou, sem interromper o notebook."""
    try:
        teste()
    except NotImplementedError:
        print(f"{nome}: ainda não resolvido")
    except AssertionError as erro:
        print(f"{nome}: confira. {erro}")
    except Exception as erro:
        print(f"{nome}: a solução deu erro. {type(erro).__name__}: {erro}")
    else:
        print(f"{nome}: ok")

### Essencial

**Exercício 2.1.** Na Listagem 2.1, calcule também o desvio no dia 900. Qual valor a lei
da raiz do tempo prevê? Guarde o desvio simulado em `desvio_900` e o previsto em
`previsto_900`.

In [ ]:
# Exercício 2.1: a Listagem 2.1 simula só 400 dias. Simule 900, com a
# mesma semente, e calcule o desvio no dia 900.

desvio_900 = None     # troque None pelo desvio simulado
previsto_900 = None   # troque None pelo valor da lei da raiz do tempo

In [ ]:
def teste_21():
    if desvio_900 is None or previsto_900 is None:
        raise NotImplementedError
    assert np.isclose(previsto_900, sigma * np.sqrt(900)), (
        "o previsto é sigma vezes a raiz de 900")
    assert abs(desvio_900 / previsto_900 - 1) < 0.03, (
        f"desvio_900 = {desvio_900:.2f}: está longe do previsto; o dia 900 "
        "é a posição 899")


conferir("Exercício 2.1", teste_21)

**Exercício 2.2.** Repita a Listagem 2.3 para o pior dia de 2008 e o pior de 2020 do
S&P 500, usando em cada caso o histórico até a véspera. Quantos desvios-padrão foram? Guarde
os escores em `z_2008` e `z_2020`. Dica: `r_sp.loc["2008"].idxmin()` é o pior dia de 2008.

In [ ]:
# Exercício 2.2: use a função desvios_na_vespera da prática guiada.

z_2008 = None   # troque None pelo escore z do pior dia de 2008
z_2020 = None   # e aqui, pelo do pior dia de 2020

In [ ]:
def teste_22():
    if z_2008 is None or z_2020 is None:
        raise NotImplementedError
    for ano, obtido in [("2008", z_2008), ("2020", z_2020)]:
        dia = r_sp.loc[ano].idxmin()
        historico = r_sp.loc[:dia - pd.Timedelta(days=1)]
        esperado = (r_sp.loc[dia] - historico.mean()) / historico.std()
        assert abs(obtido - esperado) < 0.05, (
            f"o escore de {ano} não confere: use o pior dia do ano e o "
            "histórico só até a véspera")


conferir("Exercício 2.2", teste_22)

### Aplicado

**Exercício 2.3.** Faça o mesmo cálculo para a pior queda do Ibovespa entre 2000 e 2025.
Ela é mais ou menos extrema, em desvios-padrão, do que a do S&P 500 em 1987? Guarde o
escore em `z_ibov`.

In [ ]:
_, r_ibov = carregar_retornos()   # a Listagem 2.3 só guardou o S&P 500

# Exercício 2.3: r_ibov.idxmin() é o pior dia da série do Ibovespa.

z_ibov = None   # troque None pelo escore z

**Mais ou menos extrema?** *Escreva aqui a sua resposta (dê dois cliques nesta célula para editá-la).*

In [ ]:
def teste_23():
    if z_ibov is None:
        raise NotImplementedError
    dia = r_ibov.idxmin()
    historico = r_ibov.loc[:dia - pd.Timedelta(days=1)]
    esperado = (r_ibov.loc[dia] - historico.mean()) / historico.std()
    assert abs(z_ibov - esperado) < 0.05, (
        "o escore não confere: use o pior dia do Ibovespa e o histórico só "
        "até a véspera")


conferir("Exercício 2.3", teste_23)

**Exercício 2.4.** Na Listagem 2.2, aumente o número de trajetórias para 1 000 e estime a
fração de trajetórias aritméticas que ficam negativas em 5, 10 e 25 anos. Use o mesmo
critério da listagem: a trajetória conta se passou por um preço negativo até aquele
horizonte. Guarde as frações num dicionário `fracoes`, com as chaves 5, 10 e 25.

In [ ]:
# Exercício 2.4: simule 1 000 trajetórias aritméticas de 25 anos e, para
# cada horizonte, olhe só os primeiros (anos * 252) dias de cada uma.

fracoes = None   # troque None por {5: ..., 10: ..., 25: ...}

In [ ]:
def teste_24():
    if fracoes is None:
        raise NotImplementedError
    for horizonte in [5, 10, 25]:
        # probabilidade de um passeio de passo 2 sair de 100 e tocar o zero
        # até o dia t (princípio da reflexão)
        teorica = 2 * stats.norm.cdf(-100 / (2 * np.sqrt(horizonte * 252)))
        assert abs(fracoes[horizonte] - teorica) < 0.05, (
            f"a fração de {horizonte} anos ({fracoes[horizonte]:.1%}) está longe "
            "do esperado; conte as trajetórias cujo mínimo até o horizonte "
            "ficou abaixo de zero")


conferir("Exercício 2.4", teste_24)

### Desafio

**Exercício 2.5.** Usando a `stats.t` ajustada aos retornos do S&P 500 até a véspera de
19/10/1987, calcule a probabilidade da queda daquele dia. A $t$ de Student torna o evento
plausível? O que isso sugere sobre a forma das caudas? Guarde a probabilidade em `p_t`.
Dica: `stats.t.fit(dados)` devolve os graus de liberdade, a posição e a escala.

In [ ]:
# Exercício 2.5: ajuste a t ao histórico até a véspera e calcule a
# probabilidade de um retorno tão baixo quanto o de 19/10/1987.

p_t = None   # troque None pela probabilidade

**A t torna o evento plausível? O que isso sugere?** *Escreva aqui a sua resposta (dê dois cliques nesta célula para editá-la).*

In [ ]:
def teste_25():
    if p_t is None:
        raise NotImplementedError
    assert 0 < p_t < 1, f"p_t = {p_t} não é uma probabilidade"
    historico = r_sp.loc[:"1987-10-16"]
    referencia = stats.t.cdf(r_sp.loc["1987-10-19"], *stats.t.fit(historico))
    assert abs(np.log10(p_t) - np.log10(referencia)) < 0.5, (
        f"p_t = {p_t:.1e}: confira o ajuste da t ao histórico até a véspera "
        "e use a acumulada (cdf) no retorno de 19/10/1987")


conferir("Exercício 2.5", teste_25)

## Resumo da aula

- Bachelier (1900): o preço como passeio aleatório; a dispersão cresce como
  $\sigma\sqrt{t}$.
- Osborne (1959): o passeio se aplica ao logaritmo do preço; o movimento browniano
  geométrico nunca fica negativo e é a base do modelo de Black e Scholes.
- Mandelbrot (1963) e Fama (1965): as variações de preço têm caudas pesadas, incompatíveis
  com a normal.
- A queda de 1987 foi de 29 desvios-padrão: sob a normal, um evento impossível.
- Anos 1990: a física estatística e os dados abundantes dão origem à Econofísica.

**Próxima aula:** Capítulo 3 — Mercados eficientes e passeio aleatório. Se os preços fazem
um passeio aleatório, os retornos de hoje não ajudam a prever os de amanhã. Os dados
confirmam isso?